# Cobblemon Atlas｜烈空坐與超夢動作語義診斷

此 Notebook **不重新渲染、不更改任何 WebP／checkpoint**。它會從 Google Drive 既有動畫擷取 6 個代表影格，輸出接觸圖與體型、畫面佔比、來源等診斷資料；靜態未存檔者列為 missing。診斷輸出：`Cobblemon_Atlas/reports/semantic-audit/`。

In [ ]:
from pathlib import Path
import subprocess,sys,json
DRIVE_ROOT=Path('/content/drive/MyDrive/Cobblemon_Atlas')
SITE_ROOT=Path('/content/cobblemon-atlas')
BRANCH='feat/animation-manifest-scanner-phase1'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
assert (DRIVE_ROOT/'manifests/animation-actions.json').is_file()
assert (DRIVE_ROOT/'checkpoints/render-jobs.json').is_file()
print('Drive manifest and checkpoint verified')

In [ ]:
if not (SITE_ROOT/'.git').exists():
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse','--branch',BRANCH,'https://github.com/hwh0410/cobblemon-atlas.git',str(SITE_ROOT)],check=True)
subprocess.run(['git','-C',str(SITE_ROOT),'sparse-checkout','set','tools'],check=True)
subprocess.run(['git','-C',str(SITE_ROOT),'pull','--ff-only','origin',BRANCH],check=True)
assert (SITE_ROOT/'tools/audit_animation_semantics.py').is_file()
print('Git commit:',subprocess.check_output(['git','-C',str(SITE_ROOT),'rev-parse','--short','HEAD'],text=True).strip())

In [ ]:
out=DRIVE_ROOT/'reports/semantic-audit'
cmd=[sys.executable,str(SITE_ROOT/'tools/audit_animation_semantics.py'),
     '--drive-root',str(DRIVE_ROOT),
     '--manifest',str(DRIVE_ROOT/'manifests/animation-actions.json'),
     '--out',str(out)]
subprocess.run(cmd,check=True)
audit=json.loads((out/'semantic-audit.json').read_text())
for k,v in audit['jobs'].items():
    print(k,v['review'],','.join(v.get('flags',[])),
          'contact:',v.get('contact_sheet'))
print('Review report:',out/'semantic-audit.json')

## 人工審核提示

請打開 `reports/semantic-audit/` 內的 `0384_rayquaza_ground_walk_contact.png` 和 `0384_rayquaza_physical_attackold2_contact.png`，觀察模型是否完整、方向是否正確、有無突然縮小或旋轉。**這些檢查不能僅用像素分數自動定論**。

這一階段不會把之前判定 `ready` 的工作覆寫或重新標為正式通過。診斷確認後，再針對需要的 Poser 基礎姿勢與不同動畫來源設計修復渲染。